# NB1 — Grover search under realistic noise: circuits, resources, noisy success, λ_eff fit (E1–E4)
Kaggle-ready. CPU is enough (≈30–60 min for the full run on 4 cores; set `QUICK=True` for a 5-minute smoke test).
Outputs go to `/kaggle/working/results/` (or `./results/` outside Kaggle).

| Exp. | What | Output file |
|---|---|---|
| E1 | Ideal circuits n = 2…7 vs closed form (Eq. 10) | `e1_ideal_check.csv` |
| E2 | Transpiled resources on FakeSherbrooke (Eagle), FakeTorino (Heron r1), FakeFez (Heron r2) | `e2_resources.csv` |
| E3 | Noisy success probability vs k (backend noise models), λ_eff MLE fit, model tests | `e3_counts.csv`, `e3_fits.csv` |
| E4 | Empirical expected cost, cost-optimal k, speedup, bootstrap CIs; calibration-based λ prediction | `e4_cost.csv`, `e4_summary.csv` |

In [ ]:
# Kaggle: install pinned versions (versions used to produce the reported numbers)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "qiskit==2.6.1", "qiskit-aer==0.18.0", "qiskit-ibm-runtime==0.51.0"], check=False)

In [ ]:
import os, time, json, platform, numpy as np, pandas as pd
import qiskit, qiskit_aer, qiskit_ibm_runtime, scipy
QUICK = False            # True -> fewer seeds/shots for a smoke test
SHOTS = 2048 if QUICK else 8192
SEEDS = [0, 1] if QUICK else [0, 1, 2, 3, 4]
NS = [2, 3, 4, 5, 6, 7]
MODE = "v-chain"         # primary MCX synthesis (ancilla-assisted); 'noancilla' reported in E2 only
OUT = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "results"
os.makedirs(OUT, exist_ok=True)
env = dict(python=platform.python_version(), qiskit=qiskit.__version__, aer=qiskit_aer.__version__,
           ibm_runtime=qiskit_ibm_runtime.__version__, numpy=np.__version__, scipy=scipy.__version__,
           shots=SHOTS, seeds=SEEDS, mode=MODE, quick=QUICK)
json.dump(env, open(f"{OUT}/environment.json", "w"), indent=1); print(env)

## Shared library (theory, circuits, resource metrics, statistics)

In [ ]:
"""Shared code for the Grover-under-noise experiments (E1-E4).
Qiskit >= 2.0, qiskit-aer >= 0.17, qiskit-ibm-runtime >= 0.40 (fake backends)."""
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)  # QuantumCircuit.mcx(mode=...) deprecated in Qiskit 2.1; behaviour unchanged in 2.6.1
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile

# ---------- theory (Eqs. 10, 13, 18-20 of the manuscript) ----------
def theta(N, M=1):
    return np.arcsin(np.sqrt(M / N))

def p_ideal(k, N, M=1):
    return np.sin((2 * np.asarray(k) + 1) * theta(N, M)) ** 2

def p_noisy(k, N, lam, M=1):
    q = 1.0 - lam
    k = np.asarray(k, dtype=float)
    return q ** k * p_ideal(k, N, M) + (1 - q ** k) * M / N

def r_opt(N, M=1):
    return int(round(np.pi / (4 * theta(N, M)) - 0.5))

# ---------- circuits ----------
def _mcz(qc, qubits, anc, mode):
    """Multi-controlled Z on `qubits` (last one is the target of the MCX)."""
    *ctrl, tgt = qubits
    qc.h(tgt)
    if len(ctrl) == 1:
        qc.cx(ctrl[0], tgt)
    elif len(ctrl) == 2:
        qc.ccx(ctrl[0], ctrl[1], tgt)
    elif mode == "v-chain" and anc is not None and len(anc) > 0:
        qc.mcx(ctrl, tgt, ancilla_qubits=list(anc), mode="v-chain")
    else:
        qc.mcx(ctrl, tgt)  # synthesised by the transpiler (ancilla-free)
    qc.h(tgt)

def grover_circuit(n, target, k, mode="noancilla", measure=True):
    """n-qubit single-target Grover circuit with k iterations.
    Qubit i carries bit 2**i, so the measured bitstring equals format(target, f'0{n}b')
    and x1 (MSB in the manuscript convention) is qubit n-1."""
    q = QuantumRegister(n, "q")
    regs = [q]
    anc = None
    if mode == "v-chain" and n >= 4:
        anc = QuantumRegister(n - 3, "a")
        regs.append(anc)
    if measure:
        regs.append(ClassicalRegister(n, "c"))
    qc = QuantumCircuit(*regs, name=f"grover_n{n}_k{k}")
    zeros = [i for i in range(n) if not (target >> i) & 1]
    qc.h(q)
    for _ in range(k):
        # oracle O = I - 2|x0><x0|
        if zeros:
            qc.x([q[i] for i in zeros])
        _mcz(qc, list(q), anc, mode)
        if zeros:
            qc.x([q[i] for i in zeros])
        # diffusion D = 2|psi0><psi0| - I  (up to global phase -1)
        qc.h(q); qc.x(q)
        _mcz(qc, list(q), anc, mode)
        qc.x(q); qc.h(q)
    if measure:
        qc.measure(q, regs[-1])
    return qc

# ---------- resource metrics ----------
TWO_Q = ("ecr", "cz", "cx")

def resources(tqc):
    ops = tqc.count_ops()
    n2q = sum(v for g, v in ops.items() if g in TWO_Q)
    active = {tqc.find_bit(qb).index for inst in tqc.data
              if inst.operation.name not in ("barrier", "delay") for qb in inst.qubits}
    d2 = tqc.depth(filter_function=lambda inst: inst.operation.num_qubits == 2)
    return dict(depth=tqc.depth(), depth_2q=d2, n_2q=n2q,
                n_1q=sum(v for g, v in ops.items() if g in ("rz", "sx", "x")),
                n_ops=sum(v for g, v in ops.items() if g not in ("barrier", "measure", "delay")),
                phys_qubits=len(active))

# ---------- statistics ----------
def wilson(s, n, z=1.959964):
    p = s / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h

def _nll(params, ks, succ, shots, N, model):
    if model == "1p":
        lam, A = params[0], 1.0
    else:
        lam, A = params
    q = 1 - lam
    pk = A * q ** ks * p_ideal(ks, N) + (1 - A * q ** ks) / N
    pk = np.clip(pk, 1e-12, 1 - 1e-12)
    return -np.sum(succ * np.log(pk) + (shots - succ) * np.log(1 - pk))

def fit_lambda(ks, succ, shots, N, model="1p"):
    """Binomial MLE of lambda_eff (model '1p': Eq. 18) or (lambda, A) (model '2p':
    A <= 1 absorbs a k-independent SPAM/initialisation loss)."""
    from scipy.optimize import minimize, minimize_scalar
    ks, succ, shots = map(np.asarray, (ks, succ, shots))
    if model == "1p":
        r = minimize_scalar(lambda l: _nll([l], ks, succ, shots, N, "1p"),
                            bounds=(0.0, 0.999), method="bounded")
        return dict(lam=r.x, A=1.0, nll=r.fun)
    best = None
    for l0 in (0.05, 0.3, 0.7):
        r = minimize(lambda p: _nll(p, ks, succ, shots, N, "2p"), x0=[l0, 0.9],
                     bounds=[(0.0, 0.999), (0.0, 1.0)], method="L-BFGS-B")
        if best is None or r.fun < best.fun:
            best = r
    return dict(lam=best.x[0], A=best.x[1], nll=best.fun)

def expected_cost(ks, phat):
    phat = np.maximum(np.asarray(phat, float), 1e-12)
    return (np.asarray(ks) + 1) / phat

def speedup_from_cost(N, cost):
    return (N + 1) / 2 / np.min(cost)


In [ ]:
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke, FakeTorino, FakeFez
BACKENDS = {"FakeSherbrooke": FakeSherbrooke(), "FakeTorino": FakeTorino(), "FakeFez": FakeFez()}
def target_of(n):  # manuscript instance x0 = 107 at n = 7; same low-order bits for smaller n
    return 107 % 2 ** n
def kmax_of(n):
    return 2 * r_opt(2 ** n) + 2
def used_qubits(qc):  # qubits actually acted on after synthesis (Qiskit's v-chain may leave some ancillas idle)
    qd = qc.decompose(reps=4)
    return len({qd.find_bit(q).index for inst in qd.data for q in inst.qubits})

## E1 — ideal circuits reproduce Eq. (10)

In [ ]:
rows = []
for n in NS:
    N, t = 2 ** n, target_of(n)
    for mode in ["noancilla", "v-chain"]:
        for k in range(kmax_of(n) + 1):
            probs = Statevector(grover_circuit(n, t, k, mode, measure=False)).probabilities_dict()
            pm = sum(v for b, v in probs.items() if int(b[-n:], 2) == t)
            rows.append(dict(n=n, mode=mode, k=k, P_circuit=pm, P_theory=p_ideal(k, N),
                             abs_err=abs(pm - p_ideal(k, N))))
e1 = pd.DataFrame(rows); e1.to_csv(f"{OUT}/e1_ideal_check.csv", index=False)
print("max |P_circuit - P_theory| =", e1.abs_err.max())

## E2 — transpiled resources (qubits, depth, 2q depth, gate counts)

In [ ]:
rows = []
for bname, be in BACKENDS.items():
    for n in NS:
        R = r_opt(2 ** n)
        for mode in ["noancilla", "v-chain"]:
            for k in sorted({1, R}):
                for opt in [1, 3]:
                    for s in SEEDS:
                        tq = transpile(grover_circuit(n, target_of(n), k, mode), be,
                                       optimization_level=opt, seed_transpiler=s)
                        rows.append(dict(backend=bname, n=n, mode=mode, k=k, opt=opt, seed=s,
                                         logical_qubits=used_qubits(grover_circuit(n, target_of(n), 1, mode, measure=False)),
                                         **resources(tq)))
e2 = pd.DataFrame(rows); e2.to_csv(f"{OUT}/e2_resources.csv", index=False)
e2s = (e2[e2.opt == 3].groupby(["backend", "mode", "n", "k"])
       [["logical_qubits", "phys_qubits", "depth", "depth_2q", "n_2q", "n_1q"]].agg(["mean", "std"]).round(1))
e2s.to_csv(f"{OUT}/e2_resources_summary.csv"); e2s

## E3 — noisy simulation with backend noise models
`AerSimulator.from_backend` uses each fake backend's calibration snapshot (gate errors, T1/T2, readout).
Seed s fixes both the transpiler layout and the simulator RNG; optimisation level 3; MCX mode as set above.

In [ ]:
rows = []; t0 = time.time()
for bname, be in BACKENDS.items():
    sim = AerSimulator.from_backend(be)
    for n in NS:
        t = target_of(n); key = format(t, f"0{n}b")
        for s in SEEDS:
            circs = [transpile(grover_circuit(n, t, k, MODE), be, optimization_level=3, seed_transpiler=s)
                     for k in range(kmax_of(n) + 1)]
            res = sim.run(circs, shots=SHOTS, seed_simulator=1000 + s).result()
            for k, tq in enumerate(circs):
                cnt = res.get_counts(k)
                rows.append(dict(backend=bname, n=n, seed=s, k=k, shots=SHOTS, succ=cnt.get(key, 0),
                                 **resources(tq)))
        print(f"{bname} n={n} done  ({time.time()-t0:.0f}s)")
e3 = pd.DataFrame(rows); e3.to_csv(f"{OUT}/e3_counts.csv", index=False)

In [ ]:
from scipy.stats import chi2
fits = []
for (bname, n), g in e3.groupby(["backend", "n"]):
    N = 2 ** n
    agg = g.groupby("k")[["succ", "shots"]].sum().reset_index()
    ks, sc, sh = agg.k.values, agg.succ.values, agg.shots.values
    f1, f2 = fit_lambda(ks, sc, sh, N, "1p"), fit_lambda(ks, sc, sh, N, "2p")
    # goodness of fit of the 1-parameter model: binomial deviance vs saturated model
    ph = np.clip(sc / sh, 1e-12, 1 - 1e-12)
    sat = -np.sum(sc * np.log(ph) + (sh - sc) * np.log(1 - ph))
    dev1 = 2 * (f1["nll"] - sat)
    lrt = 2 * (f1["nll"] - f2["nll"])
    # bootstrap over seeds (cluster bootstrap) for lambda_eff
    rng = np.random.default_rng(n); boot = []
    seeds = g.seed.unique()
    for _ in range(300):
        pick = rng.choice(seeds, size=len(seeds), replace=True)
        gb = pd.concat([g[g.seed == s] for s in pick]).groupby("k")[["succ", "shots"]].sum()
        boot.append(fit_lambda(gb.index.values, gb.succ.values, gb.shots.values, N, "1p")["lam"])
    fits.append(dict(backend=bname, n=n, N=N, lam_1p=f1["lam"], lam_lo=np.percentile(boot, 2.5),
                     lam_hi=np.percentile(boot, 97.5), lam_2p=f2["lam"], A_2p=f2["A"],
                     deviance_1p=dev1, df=len(ks) - 1, p_gof_1p=chi2.sf(dev1, len(ks) - 1),
                     LRT_2p_vs_1p=lrt, p_LRT=chi2.sf(lrt, 1)))
e3f = pd.DataFrame(fits); e3f.to_csv(f"{OUT}/e3_fits.csv", index=False); e3f.round(4)

## E4 — expected query cost, cost-optimal k, speedup
Ĉ(k) = (k+1)/P̂(k) (Eq. 19); Ŝ = ((N+1)/2)/min_k Ĉ(k) (Eq. 20). 95% CIs: cluster bootstrap over seeds
followed by binomial resampling of shots. Predictions use λ_eff (fitted) and λ_cal (from calibration
gate errors only: 1 − Π(1 − ε_g) per iteration, the a-priori route used in the manuscript's Discussion).

In [ ]:
def lam_calibration(be, n, s):
    t = target_of(n); kk = kmax_of(n)
    logf = []
    for k in (0, kk):
        tq = transpile(grover_circuit(n, t, k, MODE), be, optimization_level=3, seed_transpiler=s)
        lf = 0.0
        for inst in tq.data:
            name = inst.operation.name
            if name in ("measure", "barrier", "delay"): continue
            qargs = tuple(tq.find_bit(q).index for q in inst.qubits)
            try:
                err = be.target[name][qargs].error or 0.0
            except Exception:
                err = 0.0
            lf += np.log1p(-min(err, 0.999999))
        logf.append(lf)
    return 1 - np.exp((logf[1] - logf[0]) / kk)

rows, summ = [], []
for (bname, n), g in e3.groupby(["backend", "n"]):
    N = 2 ** n; be = BACKENDS[bname]
    agg = g.groupby("k")[["succ", "shots"]].sum()
    ks = agg.index.values; ph = agg.succ.values / agg.shots.values
    cost = expected_cost(ks, ph); S_hat = speedup_from_cost(N, cost); kC = ks[np.argmin(cost)]
    rng = np.random.default_rng(100 + n); bS, bk = [], []
    seeds = g.seed.unique()
    for _ in range(1000):
        pick = rng.choice(seeds, size=len(seeds), replace=True)
        gb = pd.concat([g[g.seed == s] for s in pick]).groupby("k")[["succ", "shots"]].sum()
        pb = rng.binomial(gb.shots.values, gb.succ.values / gb.shots.values) / gb.shots.values
        cb = expected_cost(gb.index.values, pb); bS.append(speedup_from_cost(N, cb)); bk.append(gb.index.values[np.argmin(cb)])
    lam = e3f[(e3f.backend == bname) & (e3f.n == n)].lam_1p.item()
    lam_cal = float(np.mean([lam_calibration(be, n, int(s)) for s in seeds]))
    kgrid = np.arange(0, 4 * int(np.sqrt(N)) + 21)
    def pred(l):
        c = (kgrid + 1) / p_noisy(kgrid, N, l); return kgrid[np.argmin(c)], (N + 1) / 2 / c.min()
    kC_fit, S_fit = pred(lam); kC_cal, S_cal = pred(lam_cal)
    for k, p, c in zip(ks, ph, cost):
        lo, hi = wilson(agg.succ[k], agg.shots[k])
        rows.append(dict(backend=bname, n=n, k=k, P_hat=p, P_lo=lo, P_hi=hi, C_hat=c,
                         P_pred_fit=p_noisy(k, N, lam), P_ideal=p_ideal(k, N)))
    summ.append(dict(backend=bname, n=n, N=N, R_opt=r_opt(N), kC_emp=kC,
                     kC_boot_mode=int(pd.Series(bk).mode()[0]), S_hat=S_hat,
                     S_lo=np.percentile(bS, 2.5), S_hi=np.percentile(bS, 97.5),
                     P_hat_Ropt=ph[r_opt(N)], lam_fit=lam, kC_pred_fit=kC_fit, S_pred_fit=S_fit,
                     lam_cal=lam_cal, kC_pred_cal=kC_cal, S_pred_cal=S_cal,
                     advantage=bool(np.percentile(bS, 2.5) > 1)))
e4 = pd.DataFrame(rows); e4.to_csv(f"{OUT}/e4_cost.csv", index=False)
e4s = pd.DataFrame(summ); e4s.to_csv(f"{OUT}/e4_summary.csv", index=False)
e4s.round(3)

## Figures

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(BACKENDS), len(NS), figsize=(3.0 * len(NS), 2.6 * len(BACKENDS)), sharey=False)
for i, b in enumerate(BACKENDS):
    for j, n in enumerate(NS):
        ax = axes[i, j]; d = e4[(e4.backend == b) & (e4.n == n)]
        ax.plot(d.k, d.P_ideal, "k--", lw=1, label="ideal (Eq. 10)")
        ax.errorbar(d.k, d.P_hat, yerr=[d.P_hat - d.P_lo, d.P_hi - d.P_hat], fmt="o", ms=3, label="noisy sim.")
        ax.plot(d.k, d.P_pred_fit, "-", lw=1, label="Eq. 18, fitted λ")
        ax.axhline(1 / 2 ** n, color="grey", lw=0.6, ls=":")
        ax.set_title(f"{b}, n={n}", fontsize=8); ax.tick_params(labelsize=7)
        if i == len(BACKENDS) - 1: ax.set_xlabel("iterations k", fontsize=8)
        if j == 0: ax.set_ylabel("P(success)", fontsize=8)
axes[0, 0].legend(fontsize=6); fig.tight_layout(); fig.savefig(f"{OUT}/fig_e3_success_vs_k.png", dpi=200)